# Experiment 3: Regex Extraction and Validation

Find and check emails, URLs, phone numbers, dates and hashtags.

| Symbol | Meaning |
|---|---|
| `\d` `\w` `\s` | digit, word char, space |
| `+` `*` `?` `{m,n}` | repeat 1+, 0+, 0-1, m to n |
| `[ ]` | any one of these chars |
| `\b` | word boundary |

- **Extraction** = `findall` (find inside text). **Validation** = `fullmatch` (whole string must match).
- Regex only checks the *form*: `31/02/2026` looks fine, so we also check the calendar.

In [ ]:
import re
from datetime import datetime

In [ ]:
text = """Contact nlp.lab@mitadt.edu.in or support-team@example.co.uk.
Visit https://www.mituniversity.ac.in/programs or www.python.org/downloads.
Call +91 98765 43210, 9123456789 or 020-30253000.
Workshop on 15/11/2026, registration closes 2026-11-05.
Follow #NLPWorkshop2026 and #MachineLearning!
Noisy: user@@mail.com, 12345, #2026, 31/02/2026"""

## Extraction patterns

In [ ]:
patterns = {
    'Email':   r'\b[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}\b',        # name @ domain . tld
    'URL':     r'(?:https?://|www\.)[^\s,]+[^\s,.]',          # starts with http(s):// or www.
    'Phone':   r'(?:\+91[\s-]?)?[6-9]\d{4}\s?\d{5}|0\d{2,4}-\d{6,8}',  # mobile or landline
    'Date':    r'\b\d{1,2}/\d{1,2}/\d{4}\b|\b\d{4}-\d{2}-\d{2}\b',    # dd/mm/yyyy or yyyy-mm-dd
    'Hashtag': r'#\w*[A-Za-z]\w*',                            # # + word with at least one letter
}

for name, p in patterns.items():
    print(f'{name:8}', re.findall(p, text))

## Validation (whole value must match)

In [ ]:
def valid_date(s):
    # Regex shape check + real calendar check
    if not re.fullmatch(patterns['Date'], s):
        return False
    for fmt in ('%d/%m/%Y', '%Y-%m-%d'):
        try:
            datetime.strptime(s, fmt)
            return True
        except ValueError:
            pass
    return False

def validate(kind, value):
    if kind == 'Date':
        return valid_date(value)
    return re.fullmatch(patterns[kind], value) is not None

tests = [('Email', 'abc.def@mail.co', True), ('Email', 'abc@mail', False),
         ('URL', 'https://example.com/a', True), ('URL', 'htp//example.com', False),
         ('Phone', '9876543210', True), ('Phone', '5876543210', False),
         ('Date', '15/11/2026', True), ('Date', '31/02/2026', False),
         ('Hashtag', '#NLP2026', True), ('Hashtag', '#2026', False)]

correct = 0
for kind, value, expected in tests:
    got = validate(kind, value)
    correct += got == expected
    print(f'{kind:8} {value:25} expected={expected!s:5} got={got}')
print(f'\nAccuracy: {correct}/{len(tests)}')

## Masking private data with `re.sub`

In [ ]:
masked = re.sub(patterns['Email'], '[EMAIL]', text)
masked = re.sub(patterns['Phone'], '[PHONE]', masked)
print(masked)